Language: **English (default)** | [Polski](lab.ipynb)

# Lab 03 -- backpropagation from scratch in NumPy

Companion material to [`README.md`](README.md). You implement forward and
backward passes for the network `Linear(1, hidden) -> tanh ->
Linear(hidden, 1)` entirely in NumPy, verify the gradients against PyTorch
`autograd`, and train a nonlinear regression with your own gradients.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch  # noqa: F401 - used in gradient_check (section 4)

rng = np.random.default_rng(42)

## 1. Data: nonlinear regression (given)

$y = \sin(2x) + 0.3x^2 + \varepsilon$ -- a clearly nonlinear function.

In [ ]:
N = 150
X = rng.uniform(-3, 3, size=(N, 1))
noise = rng.normal(0, 0.2, size=(N, 1))
Y = np.sin(2 * X) + 0.3 * X**2 + noise

plt.figure(figsize=(5, 4))
plt.scatter(X, Y, s=15, alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data: nonlinear regression")
plt.tight_layout()
plt.show()

## 2. Forward pass -- your task

`forward(X, W1, b1, W2, b2)` for `Linear(1, hidden) -> tanh ->
Linear(hidden, 1)`. Return all the intermediate values: `Z1, A1, Z2`.

In [ ]:
def forward(X, W1, b1, W2, b2):
    """Returns (Z1, A1, Z2). X: (N,1), W1: (hidden,1), W2: (1,hidden)."""
    # TODO: Z1 = X @ W1.T + b1 ; A1 = tanh(Z1) ; Z2 = A1 @ W2.T + b2
    raise NotImplementedError("TODO: forward")

## 3. Backward pass -- your task

This is the lab's core: `backward(X, Y, Z1, A1, Z2, W2)` returns `(dW1,
db1, dW2, db2)`. Use the formulas from the lecture (the *Batched version*
section), substituting the **tanh** derivative ($1-\tanh^2(z)$) instead of
sigmoid.

In [ ]:
def backward(X, Y, Z1, A1, Z2, W2):
    """Returns (dW1, db1, dW2, db2), shaped like (W1, b1, W2, b2)."""
    # TODO:
    # 1. Delta2 = dL/dZ2 (MSE, account for averaging over the batch: formula
    #    from the lecture)
    # 2. dW2 = Delta2.T @ A1 ; db2 = sum of Delta2 over the batch
    # 3. dA1 = Delta2 @ W2 ; Delta1 = dA1 * tanh'(Z1)  [tanh'(z) = 1 - tanh(z)**2]
    # 4. dW1 = Delta1.T @ X ; db1 = sum of Delta1 over the batch
    raise NotImplementedError("TODO: backward")

## 4. Gradient verification -- your task

Build the same forward pass in PyTorch with `requires_grad=True`, call
`backward()`, compare with the gradients from your own `backward`
function.

In [ ]:
def gradient_check(X, Y, W1, b1, W2, b2):
    """Returns dict {"dW1","db1","dW2","db2"} -> max|numpy - torch|."""
    # TODO:
    # 1. compute your own gradients: forward(...) then backward(...)
    # 2. build an equivalent forward pass in torch with W1,b1,W2,b2 as
    #    torch.tensor(..., requires_grad=True); torch.tensor(X), torch.tensor(Y)
    # 3. compute the MSE loss in torch, call .backward()
    # 4. return the maximum absolute difference for each of the 4 gradients
    raise NotImplementedError("TODO: gradient_check")

In [ ]:
hidden_size = 16
W1_init = rng.normal(0, 0.5, size=(hidden_size, 1))
b1_init = np.zeros(hidden_size)
W2_init = rng.normal(0, 0.5, size=(1, hidden_size))
b2_init = np.zeros(1)

diffs = gradient_check(X, Y, W1_init, b1_init, W2_init, b2_init)
for name, diff in diffs.items():
    print(f"{name}: max|numpy - torch| = {diff:.2e}")
assert max(diffs.values()) < 1e-8, (
    "gradient_check did not pass - check backward() before training"
)
print("\nOK: the gradients agree with PyTorch autograd.")

## 5. Training -- your task

`train(X, Y, hidden_size, lr, epochs)`: initializes weights randomly,
trains with your own `forward`/`backward`, updates the parameters manually
(no `torch.optim`), returns `(W1, b1, W2, b2, loss_history)`.

In [ ]:
def train(X, Y, hidden_size, lr, epochs, seed=0):
    """Returns (W1, b1, W2, b2, loss_history)."""
    # TODO:
    # 1. initialize W1 (hidden,1), b1 (hidden,), W2 (1,hidden), b2 (1,) randomly
    #    (e.g. rng.normal(0, 0.5, size=...), rng = np.random.default_rng(seed))
    # 2. in a loop over epochs: forward -> MSE loss -> backward -> update
    #    W -= lr * dW (analogously for b) -> collect the loss into a list
    raise NotImplementedError("TODO: train")

In [ ]:
W1_fit, b1_fit, W2_fit, b2_fit, loss_history = train(X, Y, hidden_size=16, lr=0.05, epochs=3000)

baseline_loss = np.mean((Y - Y.mean()) ** 2)
final_loss = loss_history[-1]
print(f"Baseline (mean) loss: {baseline_loss:.4f}")
print(f"Trained network, final loss: {final_loss:.4f}")

## 6. Interpretation -- your task

In [ ]:
x_line = np.linspace(-3, 3, 200).reshape(-1, 1)
_, _, y_line = forward(x_line, W1_fit, b1_fit, W2_fit, b2_fit)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(loss_history)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss (MSE)")
axes[0].set_title("Loss curve")

axes[1].scatter(X, Y, s=15, alpha=0.5, label="data")
axes[1].plot(x_line, y_line, color="tab:red", label="model")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Fit")
axes[1].legend()

plt.tight_layout()
plt.show()

**Task (3-5 sentences, Markdown cell):** does the loss curve look the way
you would expect after lectures 1 and 3 (shape, rate of decrease)? Does
the fit correctly capture the function's shape across the whole $x$ range,
or do you see a region with a systematic (not random) deviation from the
data -- and if not, can the remaining scatter of points around the
model's curve be explained by the noise the data was generated with alone
(section 1)?

_(Your answer here.)_

## 7. A buggy gradient -- your task

A second version of `backward`, with a deliberate bug, training an
identical network on the same data.

In [ ]:
def backward_buggy(X, Y, Z1, A1, Z2, W2):
    N = X.shape[0]
    Delta2 = (2.0 / N) * (Z2 - Y)
    dW2 = Delta2.T @ A1
    db2 = Delta2.sum(axis=0)

    dA1 = Delta2 @ W2
    Delta1 = dA1  # <- the bug is here
    dW1 = Delta1.T @ X
    db1 = Delta1.sum(axis=0)
    return dW1, db1, dW2, db2


def train_with_backward_fn(X, Y, hidden_size, lr, epochs, backward_fn, seed=0):
    g = np.random.default_rng(seed)
    W1 = g.normal(0, 0.5, size=(hidden_size, 1))
    b1 = np.zeros(hidden_size)
    W2 = g.normal(0, 0.5, size=(1, hidden_size))
    b2 = np.zeros(1)

    loss_history = []
    for _ in range(epochs):
        Z1, A1, Z2 = forward(X, W1, b1, W2, b2)
        loss_history.append(np.mean((Z2 - Y) ** 2))
        dW1, db1, dW2, db2 = backward_fn(X, Y, Z1, A1, Z2, W2)
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
    return W1, b1, W2, b2, loss_history


_, _, _, _, loss_history_correct = train_with_backward_fn(
    X, Y, hidden_size=16, lr=0.05, epochs=3000, backward_fn=backward
)
_, _, _, _, loss_history_buggy = train_with_backward_fn(
    X, Y, hidden_size=16, lr=0.05, epochs=3000, backward_fn=backward_buggy
)

plt.figure(figsize=(5.5, 4))
plt.plot(loss_history_correct, label="backward (correct)")
plt.plot(loss_history_buggy, label="backward_buggy")
plt.axhline(baseline_loss, color="black", linestyle="--", linewidth=0.8, label="baseline (mean)")
plt.xlabel("epoch")
plt.ylabel("loss (MSE)")
plt.title("Effect of a buggy gradient on training")
plt.legend()
plt.tight_layout()
plt.show()

print(f"correct backward, final loss: {loss_history_correct[-1]:.4f}")
print(f"backward_buggy,    final loss: {loss_history_buggy[-1]:.4f}")
print(f"baseline (mean):                {baseline_loss:.4f}")

**Task (3-5 sentences, Markdown cell):** identify what is missing in
`backward_buggy` compared to `backward` (refer to a specific formula from
the lecture -- which multiplication was skipped) and explain why exactly
this omission leads to the observed loss-curve behavior.

_(Your answer here.)_